In [139]:
import pandas as pd
import numpy as np


In [140]:
housing_train = pd.read_csv('train.csv')
housing_test = pd.read_csv('test.csv')

X = housing_train.drop(columns=['Id', 'SalePrice'], errors='ignore')
y = housing_train['SalePrice']
X_test = housing_test.drop(columns=['Id'], errors='ignore')

text_cols = X.select_dtypes(include=['object']).columns.tolist()


In [141]:
for column in X.columns:
    if X[column].isnull().any():
        if X[column].dtype in ['int64', 'float64']:
            fill_num = X[column].median()
            X[column] = X[column].fillna(fill_num)
            X_test[column] = X_test[column].fillna(fill_num)
        else:
            fill_txt = X[column].mode()
            X[column] = X[column].fillna(fill_txt)
            X_test[column] = X_test[column].fillna(fill_txt)

for column in X_test.columns:
    if X_test[column].isnull().any():
        if X_test[column].dtype in ['int64', 'float64']:
            X_test[column] = X_test[column].fillna(X[column].median())
        else:
            X_test[column] = X_test[column].fillna(X[column].mode())


In [142]:
X = pd.DataFrame(X).fillna(0)
X_test = pd.DataFrame(X_test).fillna(0)


In [143]:
from sklearn.preprocessing import LabelEncoder


In [144]:
X_label = X.copy()
X_test_label = X_test.copy()
le = LabelEncoder()

for col in text_cols:
    X_label[col] = le.fit_transform(X_label[col].astype(str))
    X_test_label[col] = X_test_label[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
    X_test_label[col] = le.transform(X_test_label[col])

X_label_features = X_label[text_cols].add_prefix('lbl_')
X_test_label_features = X_test_label[text_cols].add_prefix('lbl_')


In [145]:
from sklearn.preprocessing import OneHotEncoder


In [146]:
X = pd.DataFrame(X)
X_test = pd.DataFrame(X_test)


for col in text_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)


X = X.fillna(0)
X_test = X_test.fillna(0)


In [158]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [159]:
dummy_enc = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_dummy = pd.DataFrame(dummy_enc.fit_transform(X[text_cols]), index=X.index).add_prefix('dummy_')
X_test_dummy = pd.DataFrame(dummy_enc.transform(X_test[text_cols]), index=X_test.index).add_prefix('dummy_')


In [160]:
X_binary_list = []
X_test_binary_list = []

for col in text_cols:
    max_val = X_label[col].max()
    num_bits = max(1, int(np.ceil(np.log2(max_val + 1))))
    
    for i in range(num_bits):
        X_binary_list.append(pd.Series(((X_label[col].astype(int) // (2**i)) % 2), name=f'bin_{col}_{i}'))
        X_test_binary_list.append(pd.Series(((X_test_label[col].astype(int) // (2**i)) % 2), name=f'bin_{col}_{i}'))

X_binary = pd.concat(X_binary_list, axis=1) if X_binary_list else pd.DataFrame(index=X.index)
X_test_binary = pd.concat(X_test_binary_list, axis=1) if X_test_binary_list else pd.DataFrame(index=X_test.index)


In [161]:
X_target = X.copy()
X_test_target = X_test.copy()

for col in text_cols:
    target_map = housing_train.groupby(col)['SalePrice'].mean().to_dict()
    X_target[col] = X_target[col].map(target_map)
    X_test_target[col] = X_test_target[col].map(target_map).fillna(y.mean())

X_target_features = X_target[text_cols].add_prefix('target_')
X_test_target_features = X_test_target[text_cols].add_prefix('target_')


In [162]:
from sklearn.model_selection import train_test_split, KFold


In [163]:
X_kfold_features = pd.DataFrame(index=X.index)
X_test_kfold_features = pd.DataFrame(index=X_test.index)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in text_cols:
    X_kfold_features[f'kfold_{col}'] = np.nan
    
    for train_idx, val_idx in kf.split(X):
        X_tr = housing_train.iloc[train_idx]
        X_va = housing_train.iloc[val_idx]
        fold_map = X_tr.groupby(col)['SalePrice'].mean().to_dict()
        X_kfold_features.iloc[val_idx, X_kfold_features.columns.get_loc(f'kfold_{col}')] = X_va[col].map(fold_map).values
        
    global_map = housing_train.groupby(col)['SalePrice'].mean().to_dict()
    X_test_kfold_features[f'kfold_{col}'] = X_test[col].map(global_map)

X_kfold_features = X_kfold_features.fillna(y.mean())
X_test_kfold_features = X_test_kfold_features.fillna(y.mean())


In [164]:
X_numeric = X.select_dtypes(exclude=['object'])
X_test_numeric = X_test.select_dtypes(exclude=['object'])

X_final = pd.concat([X_numeric, X_label_features, X_ohe, X_dummy, X_binary, X_target_features, X_kfold_features], axis=1)
X_test_final = pd.concat([X_numeric, X_test_label_features, X_test_ohe, X_test_dummy, X_test_binary, X_test_target_features, X_test_kfold_features], axis=1)

X_final.columns = X_final.columns.astype(str)
X_test_final.columns = X_test_final.columns.astype(str)

X_train, X_val, y_train, y_val = train_test_split(X_final, y, random_state=42, test_size=0.2)


In [165]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [166]:

logistic_classifier = LogisticRegression(max_iter=100)

train_price_bins = pd.qcut(y_train, q=3, labels=['Low', 'Medium', 'High'])
validation_price_bins = pd.qcut(y_val, q=3, labels=['Low', 'Medium', 'High'])

logistic_classifier.fit(X_train_clean, train_price_bins)
validation_predictions = logistic_classifier.predict(X_val_clean)

classification_accuracy = accuracy_score(validation_price_bins, validation_predictions)
print(f"Logistic Regression Validation Accuracy: {classification_accuracy * 100:.2f}%")


Logistic Regression Validation Accuracy: 73.63%


In [167]:
from sklearn.linear_model import LinearRegression


In [168]:
X_numeric = X.select_dtypes(exclude=['object'])
X_test_numeric = X_test.select_dtypes(exclude=['object'])

X_final = pd.concat([X_numeric, X_label_features, X_ohe, X_dummy, X_binary, X_target_features, X_kfold_features], axis=1)
X_test_final = pd.concat([X_test_numeric, X_test_label_features, X_test_ohe, X_test_dummy, X_test_binary, X_test_target_features, X_test_kfold_features], axis=1)

X_final.columns = X_final.columns.astype(str)
X_test_final.columns = X_test_final.columns.astype(str)

X_final = X_final.fillna(0)
X_test_final = X_test_final.fillna(0)

X_train, X_val, y_train, y_val = train_test_split(X_final, y, random_state=42, test_size=0.2)



In [170]:

linear_regression_model = LinearRegression()
linear_regression_model.fit(X_train, y_train)

final_computed_prices = linear_regression_model.predict(X_test_final)
final_computed_prices = np.clip(final_computed_prices, 0, None)


submission_dataframe = pd.DataFrame({
    'Id': housing_test['Id'],
    'SalePrice': final_computed_prices
})


submission_dataframe.to_csv('house_prices.csv', index=False)
print("File 'house_price.csv' successfully generated from the Linear Regression model! Preview:")
print(submission_dataframe.head())


File 'house_price.csv' successfully generated from the Linear Regression model! Preview:
     Id      SalePrice
0  1461  177326.866845
1  1462  224195.432229
2  1463  132489.084442
3  1464  134444.030908
4  1465  180195.112830
